# 🧪 OmniFile AI Processor v3.0 — Testing & Debugging Notebook

## 📋 الغرض من هذا الدفتر

دفتر جوجل كولاب شامل لاختبار وتصحيح أخطاء جميع وحدات **OmniFile AI Processor v3.0**:

| الوحدة | الوظيفة |
|--------|--------|
| 🔍 **OCR Engines** | اختبار 4 محركات (TrOCR, EasyOCR, Tesseract, PaddleOCR) |
| 📊 **Fusion Strategies** | مقارنة 4 استراتيجيات دمج النتائج |
| 📝 **NLP Pipeline** | اختبار التصحيح الإملائي والترجمة والتلخيص |
| 📐 **Layout & Tables** | تحليل التخطيط واستخراج الجداول |
| 🔐 **Security** | فحص PII والتشفير |
| 📏 **Evaluation** | قياس CER/WER على بيانات الاختبار |
| 🚀 **Performance** | قياس سرعة الاستدلال واستهلاك الذاكرة |
| 🐛 **Debugging** | أدوات تشخيص الأخطاء الشاملة |

In [ ]:
# ============================================================
# الخطوة 0: معلومات النظام والبيئة
# ============================================================

import sys
import os
import platform
import subprocess
from datetime import datetime

print("=" * 60)
print("🧪 OmniFile AI Processor v3.0 — Testing & Debugging")
print("=" * 60)
print(f"📅 التاريخ: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🐍 Python: {sys.version}")
print(f"💻 النظام: {platform.system()} {platform.release()}")
print(f"🖥️ المعالج: {platform.processor()}")

# فحص GPU
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"🎮 GPU: ✅ {gpu_name} ({gpu_mem:.1f} GB)")
    else:
        print("🎮 GPU: ❌ غير متوفر")
except ImportError:
    print("🎮 GPU: ⚠️ PyTorch غير مثبت")
    gpu_available = False

# فحص CUDA
try:
    cuda_version = subprocess.check_output(["nvcc", "--version"], stderr=subprocess.DEVNULL).decode()
    print(f"⚡ CUDA: ✅")
except:
    print(f"⚡ CUDA: ⚠️ غير متوفر")

# ذاكرة النظام
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"💾 RAM: {ram.total / 1e9:.1f} GB (متاح: {ram.available / 1e9:.1f} GB)")
except ImportError:
    print("💾 RAM: psutil غير مثبت")

print("=" * 60)

## الخطوة 1: استنساخ المشروع وتثبيت الاعتماديات

In [ ]:
# ============================================================
# الخطوة 1: استنساخ المشروع وتثبيت الاعتماديات
# ============================================================

import os

# تغيير المجلد
WORK_DIR = "/content/OmniFile_Processor"

if not os.path.exists(WORK_DIR):
    print("📥 جاري استنساخ المشروع...")
    !git clone <YOUR_REPO_URL> /content/OmniFile_Processor
    os.chdir(WORK_DIR)
else:
    print("✅ المشروع موجود بالفعل")
    os.chdir(WORK_DIR)

print(f"📂 مجلد العمل: {os.getcwd()}")
print(f"📁 المحتويات: {os.listdir('.')}")

In [ ]:
# ============================================================
# تثبيت الاعتماديات (مع تخطي المحركات غير المطلوبة)
# ============================================================

print("📦 جاري تثبيت الاعتماديات...")

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft datasets huggingface_hub sentencepiece
!pip install -q easyocr pytesseract paddlepaddle paddleocr
!pip install -q opencv-python-headless Pillow PyMuPDF pdf2image
!pip install -q fastapi uvicorn numpy pandas scikit-learn
!pip install -q pyspellchecker arabic-reshaper python-bidi langdetect
!pip install -q rapidfuzz jiwer scikit-image
!pip install -q openpyxl python-docx fpdf2 beautifulsoup4
!pip install -q presidio-analyzer presidio-anonymizer detect-secrets
!pip install -q cryptography onnxruntime psutil tqdm
!pip install -q ar-corrector language-tool-python
!pip install -q pydantic pydantic-settings requests python-multipart

# تثبيت Tesseract (للنظام)
!apt-get install -q -y tesseract-ocr tesseract-ocr-arab tesseract-ocr-eng

print("\n✅ تم تثبيت جميع الاعتماديات!")

## الخطوة 2: فحص توفر المحركات والمكتبات

In [ ]:
# ============================================================
# الخطوة 2: فحص توفر جميع المحركات والمكتبات
# ============================================================

import importlib

def check_lib(name, package=None):
    """فحص توفر مكتبة"""
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'N/A')
        return True, ver
    except ImportError:
        return False, None

libraries = {
    'PyTorch': ('torch', None),
    'Transformers': ('transformers', None),
    'EasyOCR': ('easyocr', None),
    'PaddleOCR': ('paddleocr', None),
    'OpenCV': ('cv2', 'opencv-python-headless'),
    'PIL': ('PIL', 'Pillow'),
    'PyMuPDF': ('fitz', 'PyMuPDF'),
    'pdf2image': ('pdf2image', None),
    'ONNX Runtime': ('onnxruntime', None),
    'arabic_reshaper': ('arabic_reshaper', None),
    'bidi': ('bidi', 'python-bidi'),
    'langdetect': ('langdetect', None),
    'Presidio': ('presidio_analyzer', None),
    'cryptography': ('cryptography', None),
    'psutil': ('psutil', None),
    'Pydantic': ('pydantic', None),
    'FastAPI': ('fastapi', None),
    'scikit-learn': ('sklearn', 'scikit-learn'),
    'jiwer': ('jiwer', None),
    'rapidfuzz': ('rapidfuzz', None),
}

print("📋 فحص المكتبات والاعتماديات")
print("=" * 50)

available = 0
for name, (mod, pkg) in libraries.items():
    ok, ver = check_lib(mod)
    status = f"✅ v{ver}" if ok else "❌ غير مثبت"
    print(f"  {name:20s} {status}")
    if ok:
        available += 1

# فحص Tesseract
import shutil
tesseract_path = shutil.which('tesseract')
tesseract_status = f"✅ {tesseract_path}" if tesseract_path else "❌ غير مثبت"
print(f"  {'Tesseract':20s} {tesseract_status}")
if tesseract_path:
    available += 1

print(f"\n📊 الإجمالي: {available}/{len(libraries)+1} مكتبة متوفرة")
print("=" * 50)

## الخطوة 3: إعداد الإعدادات وتهيئة النظام

In [ ]:
# ============================================================
# الخطوة 3: إعداد الإعدادات
# ============================================================

import sys
sys.path.insert(0, WORK_DIR)

from config import OmniFileConfig

# إعداد البيئة
config = OmniFileConfig.from_colab_drive() if os.path.exists('/content/drive/MyDrive') else OmniFileConfig()
config.environment = "colab"
config.ensure_dirs()
config.setup_environment()

print("⚙️ الإعدادات")
print(f"  البيئة: {config.environment}")
print(f"  OCR: TrOCR={config.enable_trocr}, EasyOCR={config.enable_easyocr}, Tesseract={config.enable_tesseract}, PaddleOCR={config.enable_paddleocr}")
print(f"  ONNX: {config.use_onnx}")
print(f"  Quantization: {config.use_quantization}")
print(f"  DPI: {config.dpi}")
print(f"  GPU: {config.use_gpu if hasattr(config, 'use_gpu') else 'auto'}")

## الخطوة 4: اختبار محركات OCR (كل محرك على حدة)

In [ ]:
# ============================================================
# الخطوة 4a: إنشاء صور اختبار (عربي + إنجليزي + مختلط)
# ============================================================

import numpy as np
from PIL import Image, ImageDraw, ImageFont

def create_test_image(text, filename, width=800, height=200):
    """إنشاء صورة اختبار بنص معين"""
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
    except:
        font = ImageFont.load_default()
    draw.text((30, 80), text, fill='black', font=font)
    img.save(filename)
    return filename

# إنشاء صور اختبار
os.makedirs('test_samples', exist_ok=True)

samples = {
    'arabic': 'مرحبا بالعالم - نص عربي تجريبي',
    'english': 'Hello World - This is a test text for OCR',
    'mixed': 'Arabic مرحبا and English mixed text 123',
    'numbers': '0123456789 - ١٢٣٤٥٦٧٨٩٠',
}

for name, text in samples.items():
    path = create_test_image(text, f'test_samples/{name}.png')
    print(f"✅ تم إنشاء: test_samples/{name}.png")

# عرض الصور
from IPython.display import display
for name in samples:
    img = Image.open(f'test_samples/{name}.png')
    display(img)

In [ ]:
# ============================================================
# الخطوة 4b: اختبار محرك Tesseract
# ============================================================

import pytesseract
from PIL import Image
import time

print("🔍 اختبار محرك Tesseract")
print("=" * 50)

tesseract_results = {}

for name, expected in samples.items():
    img = Image.open(f'test_samples/{name}.png')
    
    start = time.time()
    text = pytesseract.image_to_string(img, lang='ara+eng')
    elapsed = time.time() - start
    
    tesseract_results[name] = {
        'expected': expected,
        'result': text.strip(),
        'time': elapsed
    }
    
    match = "✅" if text.strip() == expected else "⚠️"
    print(f"\n{match} اختبار {name} ({elapsed:.3f}s):")
    print(f"  المتوقع: {expected}")
    print(f"  الناتج:   {text.strip()}")

In [ ]:
# ============================================================
# الخطوة 4c: اختبار محرك EasyOCR
# ============================================================

import easyocr
import time
import numpy as np

print("🔍 اختبار محرك EasyOCR")
print("=" * 50)

reader = easyocr.Reader(['ar', 'en'], gpu=gpu_available, verbose=False)

easyocr_results = {}

for name, expected in samples.items():
    img = np.array(Image.open(f'test_samples/{name}.png'))
    
    start = time.time()
    result = reader.readtext(img)
    elapsed = time.time() - start
    
    text = ' '.join([item[1] for item in result])
    easyocr_results[name] = {
        'expected': expected,
        'result': text.strip(),
        'time': elapsed,
        'details': result
    }
    
    match = "✅" if text.strip() == expected else "⚠️"
    print(f"\n{match} اختبار {name} ({elapsed:.3f}s):")
    print(f"  المتوقع: {expected}")
    print(f"  الناتج:   {text.strip()}")
    if result:
        print(f"  التفاصيل: {len(result)} منطقة")

In [ ]:
# ============================================================
# الخطوة 4d: اختبار محرك TrOCR
# ============================================================

from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import time

print("🔍 اختبار محرك TrOCR")
print("=" * 50)

# تحميل النموذج
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')
if gpu_available:
    model = model.cuda()

trocr_results = {}

for name, expected in samples.items():
    img = Image.open(f'test_samples/{name}.png').convert('RGB')
    
    start = time.time()
    pixel_values = processor(img, return_tensors='pt').pixel_values
    if gpu_available:
        pixel_values = pixel_values.cuda()
    generated_ids = model.generate(pixel_values)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    elapsed = time.time() - start
    
    trocr_results[name] = {
        'expected': expected,
        'result': text.strip(),
        'time': elapsed
    }
    
    match = "✅" if text.strip() == expected else "⚠️"
    print(f"\n{match} اختبار {name} ({elapsed:.3f}s):")
    print(f"  المتوقع: {expected}")
    print(f"  الناتج:   {text.strip()}")

In [ ]:
# ============================================================
# الخطوة 4e: اختبار محرك PaddleOCR
# ============================================================

from paddleocr import PaddleOCR
import time
import numpy as np

print("🔍 اختبار محرك PaddleOCR")
print("=" * 50)

paddle_ocr = PaddleOCR(use_angle_cls=True, lang='ar', show_log=False)

paddleocr_results = {}

for name, expected in samples.items():
    img = np.array(Image.open(f'test_samples/{name}.png'))
    
    start = time.time()
    result = paddle_ocr.ocr(img, cls=True)
    elapsed = time.time() - start
    
    text_parts = []
    if result and result[0]:
        for line in result[0]:
            text_parts.append(line[1][0])
    text = ' '.join(text_parts)
    
    paddleocr_results[name] = {
        'expected': expected,
        'result': text.strip(),
        'time': elapsed
    }
    
    match = "✅" if text.strip() == expected else "⚠️"
    print(f"\n{match} اختبار {name} ({elapsed:.3f}s):")
    print(f"  المتوقع: {expected}")
    print(f"  الناتج:   {text.strip()}")

## الخطوة 5: مقارنة النتائج واختبار استراتيجيات الدمج

In [ ]:
# ============================================================
# الخطوة 5: مقارنة شاملة لجميع المحركات
# ============================================================

import pandas as pd

print("📊 مقارنة شاملة لنتائج المحركات")
print("=" * 70)

# بناء جدول المقارنة
comparison = []
for name, expected in samples.items():
    row = {'Test': name, 'Expected': expected}
    for engine, results in [
        ('Tesseract', tesseract_results),
        ('EasyOCR', easyocr_results),
        ('TrOCR', trocr_results),
        ('PaddleOCR', paddleocr_results)
    ]:
        if name in results:
            row[f'{engine} Result'] = results[name]['result'][:40]
            row[f'{engine} Time'] = f"{results[name]['time']:.3f}s"
    comparison.append(row)

df = pd.DataFrame(comparison)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)
display(df)

In [ ]:
# ============================================================
# الخطوة 5b: اختبار استراتيجيات الدمج
# ============================================================

sys.path.insert(0, WORK_DIR)
from modules.vision.result_fusion import ResultFusion, FusionStrategy

print("🔀 اختبار استراتيجيات الدمج")
print("=" * 50)

fusion = ResultFusion()

for name, expected in samples.items():
    print(f"\n📝 اختبار: {name}")
    print(f"   المتوقع: {expected}")
    print(f"   {'-' * 50}")
    
    # جمع نتائج جميع المحركات
    engine_results = []
    for engine, results, confidence in [
        ('Tesseract', tesseract_results, 0.7),
        ('EasyOCR', easyocr_results, 0.85),
        ('TrOCR', trocr_results, 0.9),
        ('PaddleOCR', paddleocr_results, 0.8)
    ]:
        if name in results:
            engine_results.append({
                'engine': engine,
                'text': results[name]['result'],
                'confidence': confidence
            })
    
    # اختبار كل استراتيجية
    for strategy in FusionStrategy:
        try:
            fused = fusion.fuse(engine_results, strategy=strategy)
            print(f"   {strategy.value:25s}: {fused[:60]}")
        except Exception as e:
            print(f"   {strategy.value:25s}: ❌ خطأ: {e}")

## الخطوة 6: اختبار وحدة OCREngine الموحدة

In [ ]:
# ============================================================
# الخطوة 6: اختبار محرك OCR الموحد
# ============================================================

from modules.vision.ocr_engine import OCREngine
import time

print("🧠 اختبار محرك OCR الموحد")
print("=" * 50)

engine = OCREngine(
    enable_trocr=True,
    enable_easyocr=True,
    enable_tesseract=True,
    enable_paddleocr=True,
    use_gpu=gpu_available
)

print(f"\n📊 المحركات المتاحة: {engine._get_active_engines()}")

for name, expected in samples.items():
    img = Image.open(f'test_samples/{name}.png')
    
    start = time.time()
    result = engine.recognize(img, engine='all')
    elapsed = time.time() - start
    
    text = ' '.join([token.text for token in result]) if result else '(فارغ)'
    print(f"\n{'='*50}")
    print(f"📝 {name} ({elapsed:.3f}s):")
    print(f"  المتوقع: {expected}")
    print(f"  الناتج:   {text}")

## الخطوة 7: اختبار تحليل التخطيط واستخراج الجداول

In [ ]:
# ============================================================
# الخطوة 7a: اختبار تحليل التخطيط
# ============================================================

from modules.vision.layout_analyzer import LayoutAnalyzer
import numpy as np
import cv2

print("📐 اختبار تحليل التخطيط")
print("=" * 50)

analyzer = LayoutAnalyzer()

# إنشاء صورة مستند تجريبية مع نص وجدول
doc_img = Image.new('RGB', (800, 1000), 'white')
draw = ImageDraw.Draw(doc_img)

try:
    font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 20)
    font_small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
except:
    font = ImageFont.load_default()
    font_small = font

# رسم عنوان
draw.text((50, 30), 'Document Title - عنوان المستند', fill='black', font=font)

# رسم فقرة
draw.text((50, 80), 'This is a paragraph of text for testing layout analysis.', fill='black', font=font_small)
draw.text((50, 100), 'هذا نص عربي لاختبار تحليل التخطيط.', fill='black', font=font_small)

# رسم جدول بسيط
table_x, table_y = 50, 150
for i in range(4):
    y = table_y + i * 40
    draw.line([(table_x, y), (table_x + 700, y)], fill='black', width=2)
for i in range(3):
    x = table_x + i * 233
    draw.line([(x, table_y), (x, table_y + 160)], fill='black', width=2)

# كتابة نص في الخلايا
headers = ['Column 1', 'العمود الثاني', 'Column 3']
for i, h in enumerate(headers):
    draw.text((table_x + i * 233 + 10, table_y + 10), h, fill='black', font=font_small)

doc_img.save('test_samples/layout_test.png')

# تحليل التخطيط
img_array = np.array(doc_img)
blocks = analyzer.analyze(img_array)

print(f"\n📊 تم اكتشاف {len(blocks)} كتل:")
for block in blocks:
    print(f"  - {block.block_type.value:15s} | الثقة: {block.confidence:.2f} | الموقع: ({block.bbox.x}, {block.bbox.y})")

display(doc_img)

In [ ]:
# ============================================================
# الخطوة 7b: اختبار استخراج الجداول
# ============================================================

from modules.vision.table_extractor import TableExtractor
from modules.core.structure import BBox

print("📋 اختبار استخراج الجداول")
print("=" * 50)

# تحديد منطقة الجدول يدوياً
table_bbox = BBox(x=50, y=150, width=700, height=160)

extractor = TableExtractor(ocr_engine=engine)
table_data = extractor.extract_table(img_array, table_bbox)

print(f"\n📊 بيانات الجدول المستخرجة:")
for i, row in enumerate(table_data):
    print(f"  صف {i}: {row}")

## الخطوة 8: اختبار وحدة NLP (التصحيح الإملائي والترجمة)

In [ ]:
# ============================================================
# الخطوة 8: اختبار وحدة NLP
# ============================================================

print("📝 اختبار وحدة NLP")
print("=" * 50)

# 8a: التصحيح الإملائي
print("\n🔍 التصحيح الإملائي:")
try:
    from modules.nlp.spell_corrector import SpellCorrector
    corrector = SpellCorrector()
    
    test_texts = [
        ('en', 'Ths is a tst txt for spll chcking'),
        ('ar', 'هذا نصر تجريبي لاختبار التصحيح'),
    ]
    
    for lang, text in test_texts:
        corrected = corrector.correct(text, language=lang)
        print(f"  {lang}: '{text}' → '{corrected}'")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 8b: كشف اللغة
print("\n🌍 كشف اللغة:")
try:
    from modules.nlp.language_detector import LanguageDetector
    detector = LanguageDetector()
    
    lang_samples = [
        'Hello World',
        'مرحبا بالعالم',
        'Hallo Welt',
        'Mixed text مع عربي',
    ]
    
    for text in lang_samples:
        lang = detector.detect(text)
        print(f"  '{text[:30]}' → {lang}")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 8c: الترجمة
print("\n🌐 الترجمة:")
try:
    from modules.nlp.translator import TextTranslator
    translator = TextTranslator()
    
    translated = translator.translate('Hello World', source='en', target='ar')
    print(f"  EN→AR: 'Hello World' → '{translated}'")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 8d: إعادة بناء النص العربي
print("\n🔄 إعادة بناء النص العربي:")
try:
    from modules.vision.text_reconstructor import TextReconstructor
    reconstructor = TextReconstructor()
    
    words = [
        {'text': 'مستند', 'x': 0, 'y': 0, 'w': 60, 'h': 25},
        {'text': 'تجريبي', 'x': 70, 'y': 0, 'w': 60, 'h': 25},
    ]
    text = reconstructor.reconstruct(words, direction='rtl')
    print(f"  RTL: {words} → '{text}'")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

## الخطوة 9: اختبار وحدة التقييم (CER/WER)

In [ ]:
# ============================================================
# الخطوة 9: اختبار وحدة التقييم
# ============================================================

from modules.evaluation.metrics import evaluate, calculate_cer, calculate_wer

print("📏 اختبار وحدة التقييم")
print("=" * 50)

eval_tests = [
    ('Perfect match', 'مرحبا بالعالم', 'مرحبا بالعالم'),
    ('Minor errors', 'التعرف على النصوص', 'التعرف على المصوص'),
    ('Major errors', 'معالجة الصور الرقمية', 'ملالة الصور الرقمية'),
    ('English text', 'OCR Accuracy Test', 'OCR Accurcy Test'),
    ('Mixed', 'نظام OCR متقدم', 'نظا OCR متقدم'),
]

eval_results = []

for name, ref, hyp in eval_tests:
    result = evaluate(ref, hyp)
    eval_results.append({
        'Test': name,
        'CER': f"{result.cer:.2%}",
        'WER': f"{result.wer:.2%}",
        'Accuracy': f"{result.accuracy_percent:.1f}%",
        'Grade': result.quality_grade
    })
    print(f"\n📝 {name}:")
    print(f"  المرجع:  {ref}")
    print(f"  الفرضية:  {hyp}")
    print(f"  CER: {result.cer:.2%} | WER: {result.wer:.2%} | {result.quality_grade}")

print(f"\n{'='*50}")
print("\n📊 جدول النتائج:")
eval_df = pd.DataFrame(eval_results)
display(eval_df)

## الخطوة 10: اختبار وحدة الأمان

In [ ]:
# ============================================================
# الخطوة 10: اختبار وحدة الأمان
# ============================================================

print("🔐 اختبار وحدة الأمان")
print("=" * 50)

# 10a: فحص PII
print("\n🔍 فحص البيانات الشخصية الحساسة (PII):")
try:
    from modules.security.sensitive_data_scanner import SensitiveDataScanner
    scanner = SensitiveDataScanner()
    
    pii_tests = [
        'اسمي أحمد محمد وهاتفي 0912345678',
        'My email is test@example.com',
        'Credit card: 4532-1234-5678-9010',
    ]
    
    for text in pii_tests:
        result = scanner.scan_text(text)
        print(f"  '{text}' → {len(result)} كيانات حساسة")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 10b: التشفير
print("\n🔒 اختبار التشفير:")
try:
    from modules.security.encryption import FileEncryptor
    encryptor = FileEncryptor()
    
    # إنشاء ملف اختبار
    with open('test_samples/encrypt_test.txt', 'w') as f:
        f.write('هذا نص سري للتجربة')
    
    encryptor.encrypt_file('test_samples/encrypt_test.txt', 'test_password')
    print("  ✅ تم التشفير")
    
    encryptor.decrypt_file('test_samples/encrypt_test.txt.enc', 'test_password')
    print("  ✅ تم فك التشفير")
    
    with open('test_samples/encrypt_test.txt.dec', 'r') as f:
        decrypted = f.read()
    print(f"  النص الأصلي: هذا نص سري للتجربة")
    print(f"  بعد فك التشفير: {decrypted}")
    match = decrypted == 'هذا نص سري للتجربة'
    print(f"  {'✅ متطابق' if match else '❌ غير متطابق'}")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

## الخطوة 11: اختبار معالجة PDF

In [ ]:
# ============================================================
# الخطوة 11: اختبار معالجة PDF
# ============================================================

print("📄 اختبار معالجة PDF")
print("=" * 50)

import fitz  # PyMuPDF

# إنشاء ملف PDF اختبار
pdf_doc = fitz.open()
page = pdf_doc.new_page(width=595, height=842)  # A4

# إضافة نص عربي وإنجليزي
page.insert_text((50, 100), 'Document Title - عنوان المستند', fontsize=18)
page.insert_text((50, 150), 'This is a test paragraph for PDF processing.', fontsize=12)
page.insert_text((50, 180), 'هذا نص عربي لاختبار معالجة ملفات PDF.', fontsize=12)

pdf_path = 'test_samples/test_doc.pdf'
pdf_doc.save(pdf_path)
pdf_doc.close()

print(f"\n✅ تم إنشاء: {pdf_path}")

# معالجة PDF باستخدام OCREngine
try:
    start = time.time()
    results = engine.recognize_pdf(pdf_path)
    elapsed = time.time() - start
    
    print(f"\n📊 نتائج OCR لل PDF ({elapsed:.3f}s):")
    for i, page_results in enumerate(results):
        text = ' '.join([t.text for t in page_results])
        print(f"  صفحة {i+1}: {text[:100]}...")
except Exception as e:
    print(f"  ❌ خطأ: {e}")
    import traceback
    traceback.print_exc()

## الخطوة 12: اختبار الأداء والاستهلاك

In [ ]:
# ============================================================
# الخطوة 12: قياس الأداء واستهلاك الموارد
# ============================================================

import gc
import time
import psutil

def measure_performance(engine, image_path, engine_name, iterations=3):
    """قياس أداء محرك OCR"""
    img = Image.open(image_path)
    
    times = []
    memories = []
    
    for _ in range(iterations):
        gc.collect()
        
        mem_before = psutil.Process().memory_info().rss / 1e6
        start = time.time()
        
        result = engine.recognize(img, engine=engine_name)
        
        elapsed = time.time() - start
        mem_after = psutil.Process().memory_info().rss / 1e6
        
        times.append(elapsed)
        memories.append(mem_after - mem_before)
    
    return {
        'avg_time': sum(times) / len(times),
        'min_time': min(times),
        'max_time': max(times),
        'avg_mem': sum(memories) / len(memories),
    }

print("⚡ اختبار الأداء")
print("=" * 60)

perf_engines = ['tesseract', 'easyocr']  # محركات سريعة فقط

for eng in perf_engines:
    try:
        perf = measure_performance(engine, 'test_samples/arabic.png', eng)
        print(f"\n🔍 {eng.upper()}:")
        print(f"  ⏱️  الوقت: {perf['avg_time']:.3f}s (min: {perf['min_time']:.3f}s, max: {perf['max_time']:.3f}s)")
        print(f"  💾 الذاكرة: +{perf['avg_mem']:.1f} MB")
    except Exception as e:
        print(f"\n🔍 {eng.upper()}: ❌ {e}")

## الخطوة 13: أدوات التشخيص وإصلاح الأخطاء

In [ ]:
# ============================================================
# الخطوة 13: أدوات التشخيص الشاملة
# ============================================================

import traceback
import warnings

print("🐛 أدوات التشخيص")
print("=" * 60)

# 13a: فحص إعدادات Tesseract
print("\n🔍 فحص Tesseract:")
try:
    import pytesseract
    print(f"  الإصدار: {pytesseract.get_tesseract_version()}")
    langs = pytesseract.get_languages()
    print(f"  اللغات: {langs}")
    print(f"  المسار: {pytesseract.pytesseract.tesseract_cmd}")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 13b: فحص إعدادات GPU
print("\n🎮 فحص GPU:")
try:
    import torch
    print(f"  CUDA متاح: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  عدد الأجهزة: {torch.cuda.device_count()}")
        print(f"  اسم الجهاز: {torch.cuda.get_device_name(0)}")
        print(f"  ذاكرة GPU: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
        print(f"  CUDA Version: {torch.version.cuda}")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 13c: فحص الذاكرة
print("\n💾 فحص الذاكرة:")
try:
    import psutil
    proc = psutil.Process()
    mem = proc.memory_info()
    print(f"  RSS: {mem.rss / 1e6:.1f} MB")
    print(f"  VMS: {mem.vms / 1e6:.1f} MB")
    if torch.cuda.is_available():
        print(f"  GPU Memory: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
        print(f"  GPU Cached: {torch.cuda.memory_reserved() / 1e6:.1f} MB")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

# 13d: فحص بيئة HuggingFace
print("\n🤗 فحص HuggingFace:")
try:
    from huggingface_hub import HfFolder
    token = HfFolder.get_token()
    print(f"  Token: {'✅ موجود' if token else '❌ غير موجود'}")
    import os
    cache_dir = os.path.expanduser('~/.cache/huggingface')
    if os.path.exists(cache_dir):
        total_size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, dn, filenames in os.walk(cache_dir)
            for f in filenames
        )
        print(f"  حجم الكاش: {total_size / 1e9:.2f} GB")
except Exception as e:
    print(f"  ❌ خطأ: {e}")

In [ ]:
# ============================================================
# الخطوة 13e: اختبار شامل للنظام (Full Pipeline Test)
# ============================================================

print("🧪 اختبار شامل للنظام (End-to-End)")
print("=" * 60)

def full_pipeline_test(image_path, ground_truth=""):
    """اختبار كامل لخط المعالجة"""
    results = {}
    
    # 1. تحليل التخطيط
    try:
        img = np.array(Image.open(image_path))
        blocks = analyzer.analyze(img)
        results['layout'] = {'blocks': len(blocks), 'status': '✅'}
    except Exception as e:
        results['layout'] = {'error': str(e), 'status': '❌'}
    
    # 2. OCR
    try:
        img = Image.open(image_path)
        tokens = engine.recognize(img, engine='all')
        text = ' '.join([t.text for t in tokens])
        results['ocr'] = {'text': text[:50], 'tokens': len(tokens), 'status': '✅'}
    except Exception as e:
        results['ocr'] = {'error': str(e), 'status': '❌'}
    
    # 3. التصحيح الإملائي
    try:
        if results['ocr']['status'] == '✅':
            corrected = corrector.correct(text)
            results['spellcheck'] = {'corrected': corrected[:50], 'status': '✅'}
    except Exception as e:
        results['spellcheck'] = {'error': str(e), 'status': '❌'}
    
    # 4. التقييم
    if ground_truth:
        try:
            eval_result = evaluate(ground_truth, text)
            results['evaluation'] = {
                'cer': f"{eval_result.cer:.2%}",
                'wer': f"{eval_result.wer:.2%}",
                'grade': eval_result.quality_grade,
                'status': '✅'
            }
        except Exception as e:
            results['evaluation'] = {'error': str(e), 'status': '❌'}
    
    return results

# تشغيل الاختبار الشامل
for name in samples:
    print(f"\n{'='*60}")
    print(f"📝 اختبار شامل: {name}")
    print(f"{'='*60}")
    
    results = full_pipeline_test(f'test_samples/{name}.png')
    
    for step, data in results.items():
        print(f"  {step:15s} {data['status']}")
        if 'error' in data:
            print(f"    خطأ: {data['error']}")
        elif 'text' in data:
            print(f"    نص: {data['text']}")
        elif 'cer' in data:
            print(f"    CER: {data['cer']} | WER: {data['wer']} | {data['grade']}")

## الخطوة 14: اختبار ONNX Runtime والتسريع

In [ ]:
# ============================================================
# الخطوة 14: اختبار ONNX Runtime
# ============================================================

print("⚡ اختبار ONNX Runtime")
print("=" * 50)

try:
    import onnxruntime as ort
    print(f"  ONNX Runtime: ✅ v{ort.__version__}")
    print(f"  Providers: {ort.get_available_providers()}")
    
    # تحميل TrOCR كـ ONNX
    print("\n🔄 تحويل TrOCR إلى ONNX...")
    try:
        onnx_path = engine.load_trocr_onnx()
        print(f"  ✅ تم التحويل/التحميل: {onnx_path}")
        
        # اختبار الاستدلال ONNX
        import time
        img = Image.open('test_samples/english.png').convert('RGB')
        
        start = time.time()
        result = engine._recognize_trocr_onnx(img)
        elapsed = time.time() - start
        
        print(f"  ⏱️ وقت الاستدلال ONNX: {elapsed:.3f}s")
        print(f"  الناتج: {result[:60] if result else '(فارغ)'}")
    except Exception as e:
        print(f"  ⚠️ خطأ في ONNX: {e}")
except ImportError:
    print("  ❌ ONNX Runtime غير مثبت")

## الخطوة 15: تشغيل واجهة Gradio للمعاينة التفاعلية

In [ ]:
# ============================================================
# الخطوة 15: تشغيل واجهة Gradio
# ============================================================

print("🖥️ تشغيل واجهة Gradio...")
print("=" * 50)

try:
    import pyngrok
    from pyngrok import ngrok
    
    # فتح نفق
    public_url = ngrok.connect(7860)
    print(f"🌐 رابط عام: {public_url}")
    
    # تشغيل Streamlit
    !streamlit run app.py --server.port 8501 &
    
except Exception as e:
    print(f"❌ خطأ: {e}")
    print("💡 يمكنك تشغيل الواجهة محلياً باستخدام: streamlit run app.py")

## الخطوة 16: تنظيف الذاكرة وتحرير الموارد

In [ ]:
# ============================================================
# الخطوة 16: تنظيف الذاكرة
# ============================================================

import gc
import torch

print("🧹 تنظيف الذاكرة...")

# تحرير نماذج OCR
try:
    engine.unload_models()
    print("  ✅ تم تحرير نماذج OCR")
except:
    pass

# تنظيف GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  ✅ GPU Cache محرر")

# تنظيف Python
gc.collect()
print("  ✅ GC تم")

# ذاكرة نهائية
import psutil
mem = psutil.Process().memory_info()
print(f"\n💾 الذاكرة النهائية: RSS={mem.rss/1e6:.1f} MB")

print("\n✅ انتهى اختبار وتصحيح الأخطاء!")
print("=" * 50)